РТ5-61Б Паншин Максим Владимироивич РК1

Так как предложенный датасет не содержит пропусков, которые необходимо заполнить по заданию, было принято решение взять melbourn housing snapshot, который содержит пропуски, как в числовых признаках: Building Area, так и в категориальных CouncilArea

In [ ]:
import pandas as pd

In [21]:
df = pd.read_csv('train.csv')
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [22]:
drop_cols = [
    'Id',
    'Street', 'Utilities', 'LandSlope',
    'Condition2', 'Exterior2nd', 'RoofMatl',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'LowQualFinSF', '3SsnPorch', 'ScreenPorch',
    'PoolQC', 'PoolArea', 'MiscFeature', 'MiscVal',
]
df = df.drop(columns=drop_cols)

In [23]:
missing = df.isnull().sum()
pct = (missing / len(df) * 100).round(2)
col_type = df.dtypes.apply(lambda x: 'категориальный' if x == 'str' else 'числовой')
report = pd.DataFrame({
    'Количество пропусков': missing,
    'Процент пропусков': pct,
    'Тип признака': col_type
})

report[report['Количество пропусков'] > 0].sort_values('Процент пропусков', ascending=False)

,Количество пропусков,Процент пропусков,Тип признака
Alley,1369,93.77,категориальный
Fence,1179,80.75,категориальный
MasVnrType,872,59.73,категориальный
FireplaceQu,690,47.26,категориальный
LotFrontage,259,17.74,числовой
GarageType,81,5.55,категориальный
GarageYrBlt,81,5.55,числовой
GarageFinish,81,5.55,категориальный
GarageQual,81,5.55,категориальный
GarageCond,81,5.55,категориальный


Датасет имее большое количество признаков с пропусками, по заданию нужно выбрать один числовой и один категориальный признак. Можно заметить, что большинство признаком не предоставляют инетереса, так как они отсутвуют из-за того, что какой-то их соедний признак равен нулевому значению. Например При отсутствии гаража все признаки, связанные с ним пропускаются.Так как заполнить качество камина просто новой категорией юудет не так интересно, я выбрал еще один признак: MasVnrType - тип облицовки. Так что были вабраны следующие признаки: MasVnrType, FireplaceQU, LotFrontage.

In [25]:
print(df[['LotFrontage', 'FireplaceQu', 'MasVnrType']].isnull().sum())

LotFrontage    259
FireplaceQu    690
MasVnrType     872
dtype: int64


Для заполнения пропусков в числовом признаке будем использовать медиану, чтобы не учитывать возможные выбросы. Однако возьмем не просот медиану, а медиану по району, так как вероятнее всего в одном районе архитектура зданий похожа, а значит и примыкающая к дороге часть дома будет примерно одинакова.

Для заполнения категориально признака качества камина, добавим новую категорию None, чтобы не заполнять пропуски при помощи самых часто встречающихся категорий, так как это может привести к ошибкам, когда камин отсутствует, а его качество не None.

С MasVnrType все немного инетерсней. При заполнении будем смтореть на площадь облицовки. Если она равна 0, тогда ставим категорию None, а если больше 0, то выбираем моду среди домов с облицовкой.

In [28]:
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'] \
                      .transform(lambda x: x.fillna(x.median()))
df['FireplaceQu'] = df['FireplaceQu'].fillna('None')

df.loc[(df['MasVnrType'].isnull()) & (df['MasVnrArea'] == 0), 'MasVnrType'] = 'None'
mode_with_veneer = df.loc[df['MasVnrArea'] > 0, 'MasVnrType'].mode()[0]
df['MasVnrType'] = df['MasVnrType'].fillna(mode_with_veneer)

После заполнения пропусков, посчитаем оставшиеся пропуски, чтобы убедиться в работе кода

In [30]:
print(df[['LotFrontage', 'FireplaceQu', 'MasVnrType']].isnull().sum())


LotFrontage    0
FireplaceQu    0
MasVnrType     0
dtype: int64


В финале оставим следующие признаки для обучения

GrLivArea — жилая площадь над землёй,сильно коррелирует с ценой

TotalBsmtSF — суммарная площадь подвала, лучше чем отдельные площади

LotArea — площадь участка

LotFrontage — фронтаж участка, связан с престижностью улицы

OverallQual — общая оценка качества (1–10), корреляция с ценой

OverallCond — общее состояние дома

ExterQual — качество внешней отделки

KitchenQual — качество кухни

YearBuilt — год постройки

YearRemodAdd — год последнего ремонта; разница с YearBuilt говорит о свежести

BedroomAbvGr — количество спален

FullBath — полные санузлы

TotRmsAbvGrd — всего комнат

GarageCars — вместимость гаража в машинах

FireplaceQu  — качество камина; после заполнения 'None' стал полноценным ординальным признаком

MasVnrType — тип облицовки

MasVnrArea — площадь облицовки

Neighborhood — район, даёт рыночный контекст которого нет в числовых признаках

BldgType — тип здания (дом/таунхаус/дуплекс)

HouseStyle — этажность и стиль

CentralAir — центральное кондиционирование, бинарный признак с хорошей разделяющей силой